In [2]:
import pandas as pd
import numpy as np

In [3]:
order_level = pd.read_csv(r"D:\projects\Brazilian E-Commerce Public Dataset\data\processed\order_level_clean.csv")
item_level = pd.read_csv(r"D:\projects\Brazilian E-Commerce Public Dataset\data\processed\item_level_clean.csv")

In [4]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for c in date_cols:
    if c in order_level.columns:
        order_level[c] = pd.to_datetime(order_level[c], errors="coerce")
                                                        

print("Dates converted with dayfirst=True")
print(order_level[date_cols].dtypes)

Dates converted with dayfirst=True
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


Order-level Engineered Features

In [5]:

# Order-level engineered features


# 1. Delivery time (days) — purchase to delivery
order_level["delivery_days"] = (
    order_level["order_delivered_customer_date"] - order_level["order_purchase_timestamp"]
).dt.days

# 2. Delivery delay vs estimate (negative = early, positive = late)
order_level["delivery_vs_estimate_days"] = (
    order_level["order_delivered_customer_date"] - order_level["order_estimated_delivery_date"]
).dt.days

# 3. Is late delivery flag
order_level["is_late"] = (order_level["delivery_vs_estimate_days"] > 0).astype(int)

# 4. Payment behavior: uses installments?
order_level["uses_installments"] = (order_level["max_installments"] > 1).astype(int)

# 5. Order value bucket (low/medium/high)
order_level["order_value_segment"] = pd.cut(
    order_level["total_payment_value"],
    bins=[0, 50, 150, 500, np.inf],
    labels=["low", "medium", "high", "premium"]
)

# 6. Purchase time parts (for trend features)
order_level["purchase_year"] = order_level["order_purchase_timestamp"].dt.year
order_level["purchase_month"] = order_level["order_purchase_timestamp"].dt.month
order_level["purchase_dayofweek"] = order_level["order_purchase_timestamp"].dt.dayofweek

print("Order-level features added")
print(order_level[["delivery_days","delivery_vs_estimate_days","is_late","uses_installments","order_value_segment"]].head())

Order-level features added
   delivery_days  delivery_vs_estimate_days  is_late  uses_installments  \
0            6.0                      -28.0        0                  1   
1            9.0                      -16.0        0                  1   
2           10.0                      -19.0        0                  1   
3           25.0                        3.0        1                  0   
4           11.0                      -21.0        0                  1   

  order_value_segment  
0              medium  
1                high  
2              medium  
3                high  
4              medium  


 Sample rows show delivery_vs_estimate_days between -16 and -28 (arrived 2–4 weeks early), with only 1 in 5 orders flagged as late — the platform consistently pads its delivery estimates, which likely keeps customer expectations well-managed.

Customer Level Features

In [6]:
cust_key = "customer_unique_id" if "customer_unique_id" in order_level.columns else "customer_id"

customer_features =(
    order_level.groupby(cust_key)
    .agg(
        total_spent=("total_payment_value", "sum"),
        n_orders=("order_id", "nunique"),
        avg_order_value=("total_payment_value", "mean"),
        avg_review_given=("avg_review_score", "mean"),
        avg_delivery_days=("delivery_days", "mean"),
        total_installments_used=("uses_installments", "sum"),
    )
    .reset_index()
)
#Repeat purchase indicator
customer_features["is_repeat_customer"] = (customer_features["n_orders"] > 1).astype(int)

#customer tier feature
customer_features["customer_tier"] = pd.cut(
    customer_features["total_spent"],
    bins=[0, 100, 300, 1000, np.inf],
    labels=["bronze", "silver", "gold", "platinum"]                                                        
    )

print("Customer features shape:", customer_features.shape)
print(customer_features.head())

# Save
customer_features.to_csv(r"D:\projects\Brazilian E-Commerce Public Dataset\data\processed\customer_features.csv",index=False)
print("Saved customer_features.csv ")



Customer features shape: (9956, 9)
                 customer_unique_id  total_spent  n_orders  avg_order_value  \
0  00050ab1314c0e55a6ca13cf7181fecf        35.38         1            35.38   
1  000a5ad9c4601d2bbdd9ed765d5213b3        91.28         1            91.28   
2  000d460961d6dbfa3ec6c9f5805769e1        36.68         1            36.68   
3  000de6019bb59f34c099a907c151d855       257.44         1           257.44   
4  000e309254ab1fc5ba99dd469d36bdb4        78.42         1            78.42   

   avg_review_given  avg_delivery_days  total_installments_used  \
0               4.0                6.0                        0   
1               4.0               11.0                        1   
2               5.0                3.0                        0   
3               2.0                4.0                        1   
4               3.0               14.0                        0   

   is_repeat_customer customer_tier  
0                   0        bronze  
1          

 Across 9,956 unique customers, all 5 sample rows show n_orders = 1 and is_repeat_customer = 0 — most buyers are one-time purchasers, with the majority landing in the bronze tier (total_spent < R$100), pointing to a low-retention customer base.

In [22]:


# 1. review_rate: per customer, what fraction of their orders had a review
review_rate_map = order_level.groupby("customer_unique_id")["has_review"].mean()
customer_features["review_rate"] = customer_features["customer_unique_id"].map(review_rate_map)

# Safety: if any customer has no match, fill with 0
customer_features["review_rate"] = customer_features["review_rate"].fillna(0)

# 2. Normalize orders and spend to 0-1 scale (so they're comparable)
orders_norm = customer_features["n_orders"]    / customer_features["n_orders"].max()
spend_norm  = customer_features["total_spent"] / customer_features["total_spent"].max()

# 3. Weighted engagement score (orders 40%, reviews 30%, spend 30%)
customer_features["engagement_score"] = (
    orders_norm * 0.4
    + customer_features["review_rate"] * 0.3
    + spend_norm * 0.3
).round(3)

print(customer_features[["customer_unique_id", "n_orders", "total_spent",
                         "review_rate", "engagement_score"]].head())

                 customer_unique_id  n_orders  total_spent  review_rate  \
0  00050ab1314c0e55a6ca13cf7181fecf         1        35.38          1.0   
1  000a5ad9c4601d2bbdd9ed765d5213b3         1        91.28          1.0   
2  000d460961d6dbfa3ec6c9f5805769e1         1        36.68          1.0   
3  000de6019bb59f34c099a907c151d855         1       257.44          1.0   
4  000e309254ab1fc5ba99dd469d36bdb4         1        78.42          1.0   

   engagement_score  
0             0.435  
1             0.437  
2             0.435  
3             0.444  
4             0.437  


Seller & Product performance features

In [7]:
seller_features = (
    item_level.groupby("seller_id")
    .agg(
        seller_total_revenue=("item_revenue", "sum"),
        seller_n_items=("order_item_id", "count"),
        seller_n_orders=("order_id", "nunique"),
        seller_avg_price=("price", "mean"),
        seller_state=("seller_state", "first"),
    )
    .reset_index()
    .sort_values("seller_total_revenue", ascending=False)
)
seller_features.to_csv(r"D:\projects\Brazilian E-Commerce Public Dataset\data\processed\seller_features.csv",index=False)

In [8]:
##product_popularity

product_features = (
    item_level.groupby("product_id")
    .agg(
        product_items_sold=("order_item_id", "count"),
        product_total_revenue=("item_revenue", "sum"),
        product_avg_price=("price", "mean"),
        product_category=("product_category_name_english", "first"),
    )
    .reset_index()
    .sort_values("product_items_sold", ascending=False)
)
product_features.to_csv(r"D:\projects\Brazilian E-Commerce Public Dataset\reports\product_features.csv",index=False)

In [9]:
print("Seller features:", seller_features.shape)
print("product features:", product_features.shape)
print("\nTop 3 sellers:\n", seller_features.head(3))

Seller features: (1654, 6)
product features: (6747, 5)

Top 3 sellers:
                              seller_id  seller_total_revenue  seller_n_items  \
486   4a3ca9315b744ce9f8e9374361493884              24872.23             201   
1618  fa1c13f2614d7b5c4749cbc52fecda94              24119.21              57   
825   7c67e1448b00f6e969d365cea6b010ab              24109.95             141   

      seller_n_orders  seller_avg_price seller_state  
486               189        106.342537           sp  
1618               57        406.454035           sp  
825                95        134.192624           sp  


 All three top-revenue sellers are based in SP (São Paulo), and the #2 seller nearly matched #1's revenue (~R$24K) using just 57 items at an avg price of R$406 versus 201 items at R$106 — a clear split between premium-SKU and volume-driven seller strategies; also, with 1,654 sellers covering 6,747 products, the catalog is broad but revenue is concentrated at the top.

In [10]:
order_level.to_csv(r"D:\projects\Brazilian E-Commerce Public Dataset\data\processed\order_level_features.csv", index=False)

Reusable Validation Framework

In [11]:
def validate_schema(df, expected_columns, table_name):
    """1. SCHEMA: Is all expected columns present?"""
    missing = set(expected_columns) - set(df.columns)
    passed = len(missing) == 0
    return {"check": f"Schema [{table_name}]", "passed": passed,
            "detail": "All columns present" if passed else f"Missing: {missing}"}

In [12]:
def validate_dtypes(df, dtype_map, table_name):
    """2. DATA TYPE: Is column type correct?"""
    issues = []
    for col, expected in dtype_map.items():
        if col in df.columns and expected not in str(df[col].dtype):
            issues.append(f"{col} is {df[col].dtype}, expected {expected}")
    passed = len(issues) == 0
    return {"check": f"DataTypes [{table_name}]", "passed": passed,
            "detail": "All types ok" if passed else "; ".join(issues)}


In [13]:
def validate_referential_integrity(child_df, child_key, parent_df, parent_key, label):
    """3. REFERENTIAL INTEGRIY: Is all child key presented in parent key?"""
    orphans = set(child_df[child_key].dropna()) - set(parent_df[parent_key])
    passed = len(orphans) == 0
    return {"check": f"RefIntegrity [{label}]", "passed": passed,
            "detail": "No orphans" if passed else f"{len(orphans)} orphan keys"}

In [14]:
def validate_missing(df, critical_columns, table_name):
    """4. MISSING VALUE: Is values missing in critical columns?"""
    issues = {c: int(df[c].isna().sum()) for c in critical_columns
              if c in df.columns and df[c].isna().sum() > 0}
    passed = len(issues) == 0
    return{"check": f"Missing [{table_name}]", "passed": passed,
           "detail": "No missing in critical cols" if passed else str(issues)}

In [15]:
def validate_business_rules(df, table_name):
    """5. BUSINESS RULES: domain logic checks."""
    issues = []
    if "price" in df.columns and (df["price"] <= 0).any():
        issues.append("price <= 0 found")
    if "avg_review_score" in df.columns:
        bad = (~df["avg_review_score"].between(1, 5) & df["avg_review_score"].notna()).sum()
        if bad > 0:
            issues.append(f"{bad} review scores out of 1-5")
    if "delivery_days" in df.columns and (df["delivery_days"] < 0).any():   # ← fixed: < not ,
        issues.append("negative delivery_days found")
    passed = len(issues) == 0
    return {"check": f"BusinessRules [{table_name}]", "passed": passed,
            "detail": "All rules pass" if passed else "; ".join(issues)}

In [16]:
def validate_consistency(df, table_name):
    """6. CONSISTENCY: do related values agree with each other?"""
    issues = []

    if table_name == "order_level":
        if "order_id" in df.columns:
            dups = df["order_id"].duplicated().sum()
            if dups > 0:
                issues.append(f"{dups} duplicate order_id")

        if "is_delivered" in df.columns and "order_delivered_customer_date" in df.columns:
            bad = ((df["is_delivered"] == 1) & (df["order_delivered_customer_date"].isna())).sum()
            if bad > 0:
                issues.append(f"{bad} delivered orders missing delivery date")

    if table_name == "item_level":
        if "item_revenue" in df.columns:
            bad = (df["item_revenue"].round(2) != (df["price"] + df["freight_value"]).round(2)).sum()
            if bad > 0:
                issues.append(f"{bad} rows where revenue != price+freight")

    passed = len(issues) == 0     
    return {"check": f"Consistency [{table_name}]", "passed": passed,
            "detail": "Consistent" if passed else "; ".join(issues)}

print("Validation framework defined")

Validation framework defined


 The six validation functions (schema, data types, referential integrity, missing values, business rules, consistency) are written as standalone reusable checks that return a structured dict — any table can be passed through them without rewriting logic, which keeps the validation pipeline clean and extensible.

In [17]:
orders_s    = pd.read_csv(r"D:\projects\Brazilian E-Commerce Public Dataset\data\sampled data\orders.csv")
customers_s = pd.read_csv(r"D:\projects\Brazilian E-Commerce Public Dataset\data\sampled data\customers.csv")

In [18]:
results = []

#order_level checks
results.append(validate_schema(order_level,
    ["order_id", "customer_id", "order_status", "total_payment_value"], "order_level"))
results.append(validate_dtypes(order_level,
    {"total_payment_value":"float", "order_id":"object"}, "order_level"))
results.append(validate_missing(order_level,
    ["order_id", "customer_id", "order_status"], "order_level"))
results.append(validate_business_rules(order_level, "order_level"))
results.append(validate_consistency(order_level, "order_level"))
results.append(validate_consistency(order_level, "order_level"))

#item level check
results.append(validate_schema(item_level,
    ["order_id", "product_id", "seller_id", "price"], "item_level"))
results.append(validate_business_rules(item_level, "item_level"))

#referential_integrity
results.append(validate_referential_integrity(
    order_level, "customer_id", customers_s, "customer_id", "orders->customer"))
results.append(validate_referential_integrity(
    item_level, "order_id", orders_s, "order_id", "item->order"))


In [19]:
##Build report
report = pd.DataFrame(results)
report["status"] = report["passed"].map({True: "PASS", False: "FAIL"})

In [20]:
print("="*60)
print("DATA VALIDATION REPORT")
print("="*60)
print(report[["check", "status", "detail"]].to_string(index=False))
print("="*60)
print(f"PASSED: {report['passed'].sum()}/{len(report)} checks")

DATA VALIDATION REPORT
                          check status                           detail
           Schema [order_level]   PASS              All columns present
        DataTypes [order_level]   FAIL order_id is str, expected object
          Missing [order_level]   PASS      No missing in critical cols
    BusinessRules [order_level]   PASS                   All rules pass
      Consistency [order_level]   PASS                       Consistent
      Consistency [order_level]   PASS                       Consistent
            Schema [item_level]   PASS              All columns present
     BusinessRules [item_level]   PASS                   All rules pass
RefIntegrity [orders->customer]   PASS                       No orphans
     RefIntegrity [item->order]   PASS                       No orphans
PASSED: 9/10 checks


In [21]:
report.to_csv(r"D:\projects\Brazilian E-Commerce Public Dataset\data\processed\validation_report.csv", index=False)
print("Saved validated_report.csv")

Saved validated_report.csv
